# 02 — Pré-processamento

**Tech Challenge — Fase 2 | POSTECH Data Analytics | Grupo 08**

Escopo: nulos e inconsistências, definição do alvo, normalização/codificação e variáveis de entrada.

**Entradas:** `data/raw/application_record.csv`, `data/raw/credit_record.csv`.
**Saídas (em `data/processed/`):** `base_total_trat.joblib`, `credit_filtrado.joblib`, `base_modelo.joblib`, `janelas.joblib`, `preprocessamento.joblib`.

Etapas: (1) tratar o cadastro; (2) excluir clientes sem crédito; (3) definir o alvo; (4) alvos por janela de 6 e 12 meses; (5) perfis cadastrais, variáveis e pré-processador; (6) salvar.

In [ ]:
# Configuração comum a todos os notebooks do projeto
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEMENTE = 42  # semente fixa usada em todo o projeto

# Funciona rodando a partir de notebooks/ ou da raiz do repositório.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"              # arquivos brutos (não versionados)
PROCESSED = RAIZ / "data" / "processed"  # artefatos intermediários (não versionados)
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Nulos e inconsistências do cadastro

Remove os 47 IDs duplicados (mantém a primeira ocorrência), preenche a ocupação ausente (`APOSENTADO` para pensionistas e `N_INFORMADO` para os demais) e corrige os cadastros em que o número de membros da família é menor que o de filhos.

In [ ]:
#CRIA BASE DE CADASTROS TOTAL base_total_trat COM TODOS OS REGISTROS DE CLIENTES E INICIA TRATAMENTO

import pandas as pd

# Carregar a tabela application_record.
caminho = RAW / 'application_record.csv'
application_record = pd.read_csv(caminho)

print(f'Total de linhas carregadas: {len(application_record):,}')

# Remover IDs duplicados, mantendo a primeira ocorrência.
base_total_trat = (
    application_record
    .drop_duplicates(subset=['ID'], keep='first')
    .copy()
    .reset_index(drop=True)
)

linhas_excluidas = len(application_record) - len(base_total_trat)
print(f'Linhas excluídas por ID duplicado: {linhas_excluidas:,}')

# Atualizar a ocupação dos clientes classificados como Pensioner.
condicao = (
    base_total_trat['NAME_INCOME_TYPE']
    .eq('Pensioner')
    .fillna(False)
)

base_total_trat.loc[condicao, 'OCCUPATION_TYPE'] = 'APOSENTADO'

print(f'Registros preenchidos como APOSENTADO: {condicao.sum():,}')
print(f'Total de linhas em base_total_trat: {len(base_total_trat):,}')

Total de linhas carregadas: 438,557
Linhas excluídas por ID duplicado: 47
Registros preenchidos como APOSENTADO: 75,488
Total de linhas em base_total_trat: 438,510


In [ ]:
# Atualizar a ocupação dos clientes ainda não preenchidos
ocupacao = base_total_trat['OCCUPATION_TYPE'].astype('string')

condicao = (
    ocupacao.isna()
    | ocupacao.str.strip().eq('')
).fillna(False)

base_total_trat.loc[
    condicao, 'OCCUPATION_TYPE'
] = 'N_INFORMADO'

print(f'Registros preenchidos com N_INFORMADO: {condicao.sum():,}')
print(f'Total de linhas em base_total_trat: {len(base_total_trat):,}')

Registros preenchidos com N_INFORMADO: 58,841
Total de linhas em base_total_trat: 438,510


In [ ]:
#FINALIZA TRATAMENTO DA base_total_trat TRATANDO CLIENTES COM (CNT_FAM_MEMBERS) < (CNT_CHILDREN)

import pandas as pd

membros = pd.to_numeric(base_total_trat['CNT_FAM_MEMBERS'], errors='raise')
filhos = pd.to_numeric(base_total_trat['CNT_CHILDREN'], errors='raise')

condicao = membros.lt(filhos).fillna(False)

base_total_trat.loc[
    condicao, 'CNT_FAM_MEMBERS'
] = filhos.loc[condicao] + 2

print(f'Registros corrigidos: {condicao.sum():,}')

Registros corrigidos: 41


## 2. Histórico: excluir clientes que nunca tiveram crédito

Clientes cujo histórico tem somente `STATUS = X` não têm exposição a crédito e saem da base (`CREDIT_FILTRADO`). Depois, o cadastro é restrito a quem tem histórico (`base_inicial_modelo`).

In [ ]:
#CRIA TABELA INTERMEDIÁRIA CREDIT_FILTRADO QUE EXCLUI CLIENTES QUE NUNCA TIVERAM CRÉDITO DA TABELA credit_record

import pandas as pd
from IPython.display import display

credit_record = pd.read_csv(
    RAW / 'credit_record.csv',
    dtype={'ID': 'string', 'STATUS': 'string'}
)

# Identificar clientes cujo histórico contém somente X.
clientes_somente_X = (
    credit_record['STATUS']
    .eq('X')
    .fillna(False)
    .groupby(credit_record['ID'])
    .transform('all')
)

# Excluir todos os registros desses clientes.
CREDIT_FILTRADO = (
    credit_record.loc[~clientes_somente_X]
    .copy()
    .reset_index(drop=True)
)

print(f"IDs antes do filtro: {credit_record['ID'].nunique():,}")
print(
    f"IDs excluídos — somente X: "
    f"{credit_record.loc[clientes_somente_X, 'ID'].nunique():,}"
)
print(f"IDs após o filtro: {CREDIT_FILTRADO['ID'].nunique():,}")
print(f"Registros após o filtro: {len(CREDIT_FILTRADO):,}")

IDs antes do filtro: 45,985
IDs excluídos — somente X: 4,536
IDs após o filtro: 41,449
Registros após o filtro: 947,052


In [ ]:
#CRIA TABELA DE CADASTRO PARA O MODELO
# Comparar IDs como texto sem alterar as tabelas de origem.
ids_credito = (
    CREDIT_FILTRADO['ID']
    .astype('string')
    .str.strip()
    .dropna()
    .unique()
)

ids_cadastro = (
    base_total_trat['ID']
    .astype('string')
    .str.strip()
)

# Preservar todas as colunas do cadastro tratado.
base_inicial_modelo = (
    base_total_trat.loc[ids_cadastro.isin(ids_credito)]
    .copy()
    .reset_index(drop=True)
)

print(f'Linhas do cadastro tratado: {len(base_total_trat):,}')
print(f'Linhas de base_inicial_modelo: {len(base_inicial_modelo):,}')
print(f"IDs distintos na nova base: {base_inicial_modelo['ID'].nunique():,}")
print(f'Colunas preservadas: {base_inicial_modelo.shape[1]}')

display(base_inicial_modelo.head(10))

Linhas do cadastro tratado: 438,510
Linhas de base_inicial_modelo: 33,110
IDs distintos na nova base: 33,110
Colunas preservadas: 18


        ID CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  \
0  5008804           M            Y               Y             0   
1  5008805           M            Y               Y             0   
2  5008806           M            Y               Y             0   
3  5008808           F            N               Y             0   
4  5008810           F            N               Y             0   
5  5008811           F            N               Y             0   
6  5008812           F            N               Y             0   
7  5008813           F            N               Y             0   
8  5008814           F            N               Y             0   
9  5008815           M            Y               Y             0   

   AMT_INCOME_TOTAL      NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0          427500.0               Working               Higher education   
1          427500.0               Working               Higher education   
2          1

## 3. Definição do alvo

`STATUS` 2 a 5 significa atraso de 60 dias ou mais; `C` (pago) e `X` (sem empréstimo) não contam como atraso. A tabela compara os cortes de 30+ e 60+ dias e os históricos mínimos de 6 e 12 meses. O corte adotado é **60+ dias**: o evento é raro (cerca de 2%) e representa atraso grave.

`ALVO = 1` quando o cliente tem ao menos um mês com atraso de 60+ dias; exige histórico mínimo de 6 meses.

In [ ]:
#Avaliar cortes de atraso para ajudar a decidir quando o atraso é problemático

import pandas as pd
from IPython.display import display

credito = CREDIT_FILTRADO[
    ['ID', 'MONTHS_BALANCE', 'STATUS']
].copy()

# Validar dados antes da contagem.
if credito['ID'].isna().any():
    raise ValueError('Existem IDs ausentes na tabela.')

credito['MONTHS_BALANCE'] = pd.to_numeric(
    credito['MONTHS_BALANCE'], errors='raise'
)

if (
    credito['MONTHS_BALANCE'].isna().any()
    or credito['MONTHS_BALANCE'].mod(1).ne(0).any()
    or credito['MONTHS_BALANCE'].gt(0).any()
):
    raise ValueError('Existem meses ausentes ou inválidos.')

credito['STATUS'] = credito['STATUS'].astype('string').str.strip()

status_validos = ['0', '1', '2', '3', '4', '5', 'C', 'X']

if not credito['STATUS'].isin(status_validos).all():
    raise ValueError('Existem STATUS ausentes ou desconhecidos.')

if credito.duplicated(['ID', 'MONTHS_BALANCE']).any():
    raise ValueError('Existem registros duplicados de ID + mês.')

# Indicadores mensais: C e X não são classificados como atraso.
credito['ATRASO_30_MAIS'] = credito['STATUS'].isin(
    ['1', '2', '3', '4', '5']
)
credito['ATRASO_60_MAIS'] = credito['STATUS'].isin(
    ['2', '3', '4', '5']
)

# Uma linha por cliente.
RESUMO_CREDITO_CLIENTE = (
    credito.groupby('ID')
    .agg(
        MESES_REGISTRADOS=('MONTHS_BALANCE', 'nunique'),
        MESES_ATRASO_30_MAIS=('ATRASO_30_MAIS', 'sum'),
        MESES_ATRASO_60_MAIS=('ATRASO_60_MAIS', 'sum')
    )
    .reset_index()
)

# Comparar histórico mínimo × corte de atraso.
resultados = []
total_clientes = len(RESUMO_CREDITO_CLIENTE)

for minimo_meses in [6, 12]:
    elegiveis = RESUMO_CREDITO_CLIENTE.loc[
        RESUMO_CREDITO_CLIENTE['MESES_REGISTRADOS']
        .ge(minimo_meses)
    ]

    for corte in [30, 60]:
        coluna = f'MESES_ATRASO_{corte}_MAIS'

        total_elegiveis = len(elegiveis)
        clientes_atraso = elegiveis[coluna].gt(0).sum()

        resultados.append({
            'HISTORICO_MINIMO_MESES': minimo_meses,
            'CORTE_ATRASO_DIAS': f'{corte}+',
            'CLIENTES_ELEGIVEIS': total_elegiveis,
            'CLIENTES_EXCLUIDOS_HISTORICO_CURTO':
                total_clientes - total_elegiveis,
            'CLIENTES_COM_ATRASO': int(clientes_atraso),
            'CLIENTES_SEM_ATRASO_NO_CORTE':
                int(total_elegiveis - clientes_atraso),
            'PERCENTUAL_CLIENTES_COM_ATRASO': (
                round(100 * clientes_atraso / total_elegiveis, 2)
                if total_elegiveis else float('nan')
            ),
            'MESES_COM_ATRASO': int(elegiveis[coluna].sum())
        })

COMPARACAO_CORTES = pd.DataFrame(resultados)

display(COMPARACAO_CORTES)

   HISTORICO_MINIMO_MESES CORTE_ATRASO_DIAS  CLIENTES_ELEGIVEIS  \
0                       6               30+               36876   
1                       6               60+               36876   
2                      12               30+               29175   
3                      12               60+               29175   

   CLIENTES_EXCLUIDOS_HISTORICO_CURTO  CLIENTES_COM_ATRASO  \
0                                4573                 5175   
1                                4573                  659   
2                               12274                 4397   
3                               12274                  597   

   CLIENTES_SEM_ATRASO_NO_CORTE  PERCENTUAL_CLIENTES_COM_ATRASO  \
0                         31701                           14.03   
1                         36217                            1.79   
2                         24778                           15.07   
3                         28578                            2.05   

   MESES_COM_ATRA

In [ ]:
import pandas as pd
from IPython.display import display

# 1. Configuração ajustável.
CORTE_ATRASO_DIAS = 60       # Aceita 30 ou 60.
MINIMO_MESES = 6

if CORTE_ATRASO_DIAS not in [30, 60]:
    raise ValueError('Use corte de 30 ou 60 dias.')

# 2. Preparar cópias das tabelas.
credito = CREDIT_FILTRADO[['ID', 'MONTHS_BALANCE', 'STATUS']].copy()
cadastro = base_inicial_modelo.copy()

for tabela in [credito, cadastro]:
    tabela['ID'] = tabela['ID'].astype('string')
    if tabela['ID'].isna().any() or tabela['ID'].str.strip().eq('').any():
        raise ValueError('Existem IDs ausentes ou vazios.')

credito['MONTHS_BALANCE'] = pd.to_numeric(
    credito['MONTHS_BALANCE'], errors='raise'
)
credito['STATUS'] = credito['STATUS'].astype('string').str.strip()

if not credito['STATUS'].isin(['0', '1', '2', '3', '4', '5', 'C', 'X']).all():
    raise ValueError('Existem STATUS ausentes ou desconhecidos.')

if credito['MONTHS_BALANCE'].isna().any():
    raise ValueError('Existem meses ausentes.')

if credito.duplicated(['ID', 'MONTHS_BALANCE']).any():
    raise ValueError('Existem registros repetidos de ID + mês.')

if cadastro['ID'].duplicated().any():
    raise ValueError('O cadastro precisa conter uma única linha por ID.')

if 'ALVO' in cadastro.columns:
    raise ValueError('O cadastro já contém ALVO. Revise antes de continuar.')

# 3. Construir o alvo a partir de todo o histórico disponível.
status_atraso = (
    ['1', '2', '3', '4', '5']
    if CORTE_ATRASO_DIAS == 30
    else ['2', '3', '4', '5']
)

credito['ATRASO'] = credito['STATUS'].isin(status_atraso)

RESUMO_ALVO = (
    credito.groupby('ID')
    .agg(
        MESES_REGISTRADOS=('MONTHS_BALANCE', 'nunique'),
        ALVO=('ATRASO', 'max')
    )
    .reset_index()
)

RESUMO_ALVO = RESUMO_ALVO.loc[
    RESUMO_ALVO['MESES_REGISTRADOS'].ge(MINIMO_MESES)
].copy()

RESUMO_ALVO['ALVO'] = RESUMO_ALVO['ALVO'].astype(int)

# Somente cadastro + alvo; métricas do histórico não entram como preditores.
BASE_MODELO = cadastro.merge(
    RESUMO_ALVO[['ID', 'ALVO']],
    on='ID',
    how='inner',
    validate='one_to_one'
)

X = BASE_MODELO.drop(columns=['ID', 'ALVO'])
y = BASE_MODELO['ALVO']

contagens = y.value_counts()
if len(contagens) != 2 or contagens.min() < 2:
    raise ValueError('São necessários exemplos suficientes das duas classes.')

print(f'Clientes na base do modelo: {len(BASE_MODELO):,}')
print(f'Clientes com atraso (ALVO = 1): {int(y.sum()):,} ({100 * y.mean():.2f}%)')

## 4. Alvo por janela de observação (6 e 12 meses)

O modelo final prevê atraso nos **primeiros 12 meses** de histórico de cada cliente. Para isso, o alvo é recalculado em janelas fixas; clientes sem janela completa ou só com `X` na janela não são elegíveis (nunca viram classe 0). Quatro cenários são montados (6 e 12 meses, com todos os elegíveis e com a coorte comum) e conferidos entre si.

In [ ]:
#CRIAÇÃO E COMPARAÇÃO DE CENÁRIOS

import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

MINIMO_MESES_DIFERENTES_X = 1

# ============================================================
# 1. Preparar cópias dos dados
# ============================================================

cadastro = BASE_MODELO.drop(columns='ALVO').copy()
colunas_entrada = [
    coluna for coluna in cadastro.columns if coluna != 'ID'
]

credito_janelas = CREDIT_FILTRADO[
    ['ID', 'MONTHS_BALANCE', 'STATUS']
].copy()

for tabela in [cadastro, credito_janelas]:
    tabela['ID'] = tabela['ID'].astype('string').str.strip()

    if tabela['ID'].isna().any() or tabela['ID'].eq('').any():
        raise ValueError('Existem IDs ausentes ou vazios.')

if cadastro['ID'].duplicated().any():
    raise ValueError('O cadastro precisa conter uma linha por ID.')

credito_janelas['MONTHS_BALANCE'] = pd.to_numeric(
    credito_janelas['MONTHS_BALANCE'], errors='raise'
)

meses = credito_janelas['MONTHS_BALANCE']

if (
    meses.isna().any()
    or meses.mod(1).ne(0).any()
    or meses.gt(0).any()
):
    raise ValueError('Existem meses ausentes ou inválidos.')

credito_janelas['STATUS'] = (
    credito_janelas['STATUS'].astype('string').str.strip()
)

if not credito_janelas['STATUS'].isin(
    ['0', '1', '2', '3', '4', '5', 'C', 'X']
).all():
    raise ValueError('Existem STATUS ausentes ou desconhecidos.')

if credito_janelas.duplicated(['ID', 'MONTHS_BALANCE']).any():
    raise ValueError('Existem registros duplicados de ID + mês.')

# Limitar o experimento aos clientes da BASE_MODELO atual.
credito_janelas = credito_janelas.loc[
    credito_janelas['ID'].isin(cadastro['ID'])
].copy()

ids_sem_historico = set(cadastro['ID']) - set(credito_janelas['ID'])

if ids_sem_historico:
    raise ValueError('Há clientes do cadastro sem histórico correspondente.')

# Ordenar do registro mais antigo para o mais recente.
credito_janelas = credito_janelas.sort_values(
    ['ID', 'MONTHS_BALANCE']
)

primeiro_mes = (
    credito_janelas.groupby('ID')['MONTHS_BALANCE'].transform('min')
)

# Posição no calendário relativo de cada cliente:
# mês mais antigo = 0; mês seguinte = 1; etc.
credito_janelas['MES_RELATIVO'] = (
    credito_janelas['MONTHS_BALANCE'] - primeiro_mes
)

credito_janelas['DIFERENTE_X'] = (
    credito_janelas['STATUS'].ne('X')
)
credito_janelas['STATUS_NUMERICO'] = (
    credito_janelas['STATUS'].isin(['0', '1', '2', '3', '4', '5'])
)
credito_janelas['ATRASO_60_MAIS'] = (
    credito_janelas['STATUS'].isin(['2', '3', '4', '5'])
)

# ============================================================
# 2. Criar alvos nas janelas fixas
# ============================================================

JANELAS_CLIENTES = {}
diagnostico_janelas = []

for tamanho in [6, 12]:
    registros_janela = credito_janelas.loc[
        credito_janelas['MES_RELATIVO'].lt(tamanho)
    ]

    resumo = (
        registros_janela.groupby('ID')
        .agg(
            MESES_REGISTRADOS=('MONTHS_BALANCE', 'nunique'),
            PRIMEIRO_MES=('MONTHS_BALANCE', 'min'),
            ULTIMO_MES=('MONTHS_BALANCE', 'max'),
            MESES_DIFERENTES_X=('DIFERENTE_X', 'sum'),
            MESES_STATUS_NUMERICO=('STATUS_NUMERICO', 'sum'),
            MESES_ATRASO_60_MAIS=('ATRASO_60_MAIS', 'sum')
        )
        .reset_index()
    )

    # Exigir todos os meses do intervalo, sem preencher lacunas.
    resumo['JANELA_COMPLETA'] = resumo['MESES_REGISTRADOS'].eq(tamanho)
    resumo['SOMENTE_X'] = resumo['MESES_DIFERENTES_X'].eq(0)

    resumo['ELEGIVEL'] = (
        resumo['JANELA_COMPLETA']
        & resumo['MESES_DIFERENTES_X'].ge(
            MINIMO_MESES_DIFERENTES_X
        )
    )

    # Ausência de observação suficiente não vira classe 0.
    resumo['ALVO_JANELA'] = (
        resumo['MESES_ATRASO_60_MAIS'].gt(0).astype('Int64')
        .where(resumo['ELEGIVEL'], pd.NA)
    )

    JANELAS_CLIENTES[tamanho] = resumo

    elegiveis = resumo.loc[resumo['ELEGIVEL']]

    diagnostico_janelas.append({
        'JANELA_MESES': tamanho,
        'CLIENTES_NO_UNIVERSO': len(cadastro),
        'JANELAS_COMPLETAS': int(resumo['JANELA_COMPLETA'].sum()),
        'JANELAS_INCOMPLETAS': int((~resumo['JANELA_COMPLETA']).sum()),
        'COMPLETAS_SOMENTE_X': int(
            (resumo['JANELA_COMPLETA'] & resumo['SOMENTE_X']).sum()
        ),
        'CLIENTES_ELEGIVEIS': len(elegiveis),
        'CLASSE_0': int(elegiveis['ALVO_JANELA'].eq(0).sum()),
        'CLASSE_1': int(elegiveis['ALVO_JANELA'].eq(1).sum()),
        'PERCENTUAL_CLASSE_1': (
            100 * elegiveis['ALVO_JANELA'].mean()
            if len(elegiveis) else float('nan')
        )
    })

DIAGNOSTICO_JANELAS = pd.DataFrame(diagnostico_janelas)

print('DIAGNÓSTICO DAS JANELAS:')
display(DIAGNOSTICO_JANELAS.round(2))

# ============================================================
# 3. Definir os quatro cenários
# ============================================================

elegiveis_6 = JANELAS_CLIENTES[6].loc[
    JANELAS_CLIENTES[6]['ELEGIVEL']
].set_index('ID')

elegiveis_12 = JANELAS_CLIENTES[12].loc[
    JANELAS_CLIENTES[12]['ELEGIVEL']
].set_index('ID')

ids_comuns = elegiveis_6.index.intersection(elegiveis_12.index)

CENARIOS_JANELAS = {
    '6 meses — elegíveis': elegiveis_6.copy(),
    '12 meses — elegíveis': elegiveis_12.copy(),
    '6 meses — coorte comum': elegiveis_6.loc[ids_comuns].copy(),
    '12 meses — coorte comum': elegiveis_12.loc[ids_comuns].copy()
}

resumo_cenarios = []

for nome, tabela in CENARIOS_JANELAS.items():
    resumo_cenarios.append({
        'CENARIO': nome,
        'CLIENTES': len(tabela),
        'CLASSE_0': int(tabela['ALVO_JANELA'].eq(0).sum()),
        'CLASSE_1': int(tabela['ALVO_JANELA'].eq(1).sum()),
        'PERCENTUAL_CLASSE_1': (
            100 * tabela['ALVO_JANELA'].mean()
            if len(tabela) else float('nan')
        ),
        'MEDIA_MESES_DIFERENTES_X':
            tabela['MESES_DIFERENTES_X'].mean(),
        'MEDIANA_MESES_DIFERENTES_X':
            tabela['MESES_DIFERENTES_X'].median()
    })

COMPARACAO_ALVOS_JANELAS = pd.DataFrame(resumo_cenarios)

print('\nALVOS E EXPOSIÇÃO DOS QUATRO CENÁRIOS:')
display(COMPARACAO_ALVOS_JANELAS.round(2))

# Na coorte comum, comparar diretamente as mudanças do alvo.
TRANSICOES_ALVO_JANELAS = pd.crosstab(
    elegiveis_6.loc[ids_comuns, 'ALVO_JANELA'],
    elegiveis_12.loc[ids_comuns, 'ALVO_JANELA'],
    rownames=['ALVO_6_MESES'],
    colnames=['ALVO_12_MESES']
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

print('\nMUDANÇAS DO ALVO NOS MESMOS CLIENTES:')
display(TRANSICOES_ALVO_JANELAS)

# Os primeiros 6 meses estão contidos nos primeiros 12:
# um positivo em 6 meses não pode virar negativo em 12.
if TRANSICOES_ALVO_JANELAS.loc[1, 0] != 0:
    raise ValueError('Inconsistência: alvo positivo em 6 virou negativo em 12.')

DIAGNÓSTICO DAS JANELAS:


   JANELA_MESES  CLIENTES_NO_UNIVERSO  JANELAS_COMPLETAS  JANELAS_INCOMPLETAS  \
0             6                 29029              29029                    0   
1            12                 29029              22435                 6594   

   COMPLETAS_SOMENTE_X  CLIENTES_ELEGIVEIS  CLASSE_0  CLASSE_1  \
0                 1516               27513     27323       190   
1                  704               21731     21361       370   

   PERCENTUAL_CLASSE_1  
0                 0.69  
1                 1.70  


ALVOS E EXPOSIÇÃO DOS QUATRO CENÁRIOS:


                   CENARIO  CLIENTES  CLASSE_0  CLASSE_1  PERCENTUAL_CLASSE_1  \
0      6 meses — elegíveis     27513     27323       190                 0.69   
1     12 meses — elegíveis     21731     21361       370                 1.70   
2   6 meses — coorte comum     20990     20837       153                 0.73   
3  12 meses — coorte comum     20990     20632       358                 1.71   

   MEDIA_MESES_DIFERENTES_X  MEDIANA_MESES_DIFERENTES_X  
0                      5.28                         6.0  
1                     10.65                        12.0  
2                      5.29                         6.0  
3                     10.91                        12.0  


MUDANÇAS DO ALVO NOS MESMOS CLIENTES:


ALVO_12_MESES      0    1
ALVO_6_MESES             
0              20632  205
1                  0  153

## 5. Perfis cadastrais, variáveis de entrada e normalização

- **Variáveis:** todas as colunas cadastrais exceto `ID` (idade e tempo de emprego seguem em dias, como no cadastro; a conversão para anos é só da EDA). Nenhuma variável derivada foi criada.
- **Perfis:** clientes com todas as variáveis cadastrais iguais formam um perfil (`GRUPO_PERFIL`). Os perfis são usados nos notebooks seguintes para que o mesmo perfil nunca apareça no treino e no teste.
- **Normalização e codificação:** numéricas com imputação pela mediana e `StandardScaler`; categóricas com imputação `N_INFORMADO` e `OneHotEncoder`. O ajuste acontece só dentro das pipelines, em cada treino.

In [ ]:
# ============================================================
# 4. Grupos cadastrais (perfis) e matriz de entrada
# ============================================================

# Criar os grupos antes do pré-processamento.
grupos = (
    cadastro.groupby(
        colunas_entrada,
        dropna=False,
        observed=True,
        sort=False
    )
    .ngroup()
)

grupos_por_id = pd.Series(
    grupos.to_numpy(),
    index=cadastro['ID'],
    name='GRUPO_PERFIL'
)

X_cadastro = cadastro.set_index('ID')[colunas_entrada].copy()

numericas = X_cadastro.select_dtypes(
    include='number'
).columns.tolist()
categoricas = [c for c in colunas_entrada if c not in numericas]

for coluna in numericas:
    X_cadastro[coluna] = pd.to_numeric(
        X_cadastro[coluna], errors='raise'
    ).astype('float64')

for coluna in categoricas:
    valores = X_cadastro[coluna].astype('string')
    X_cadastro[coluna] = (
        valores.astype(object)
        .where(valores.notna(), float('nan'))
    )

In [ ]:
# Normalização e codificação.
# O pré-processador é apenas DEFINIDO aqui. Ele é ajustado dentro das pipelines do
# notebook 03, somente com os dados de treino de cada divisão (evita vazamento).
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_base = BASE_MODELO.drop(columns=["ID", "ALVO"])

colunas_numericas = X_base.select_dtypes(include="number").columns.tolist()
colunas_categoricas = [c for c in X_base.columns if c not in colunas_numericas]

pipeline_numerica = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("padronizacao", StandardScaler())
])

pipeline_categorica = Pipeline([
    ("imputacao", SimpleImputer(
        strategy="constant",
        fill_value="N_INFORMADO"
    )),
    ("codificacao", OneHotEncoder(handle_unknown="ignore"))
])

preprocessador = ColumnTransformer([
    ("numericas", pipeline_numerica, colunas_numericas),
    ("categoricas", pipeline_categorica, colunas_categoricas)
])

print(f"Variáveis numéricas ({len(colunas_numericas)}): {colunas_numericas}")
print(f"Variáveis categóricas ({len(colunas_categoricas)}): {colunas_categoricas}")

## 6. Salvar artefatos

Resultado do pré-processamento: base tratada, base de modelagem (cadastro + `ALVO`), janelas, perfis e o pré-processador. Os notebooks 03 e 04 só leem estes arquivos.

> Resumo (Colab): 438.510 cadastros únicos; 75.488 registros com ocupação `APOSENTADO` e 58.841 `N_INFORMADO`; 41 correções de membros da família; 4.536 clientes só com `X` excluídos (41.449 restantes); base de modelagem de 29.029 clientes (608 com atraso 60+, 2,1%); janela de 12 meses com 21.731 elegíveis (370 com atraso, 1,70%).

In [ ]:
# Salvar os artefatos usados pelos notebooks 03 e 04.
joblib.dump(base_total_trat, PROCESSED / "base_total_trat.joblib")
joblib.dump(CREDIT_FILTRADO, PROCESSED / "credit_filtrado.joblib")
joblib.dump(BASE_MODELO, PROCESSED / "base_modelo.joblib")

joblib.dump(
    {
        "JANELAS_CLIENTES": JANELAS_CLIENTES,
        "CENARIOS_JANELAS": CENARIOS_JANELAS,
        "elegiveis_6": elegiveis_6,
        "elegiveis_12": elegiveis_12,
        "ids_comuns": ids_comuns,
        "X_cadastro": X_cadastro,
        "grupos_por_id": grupos_por_id,
        "colunas_entrada": colunas_entrada,
    },
    PROCESSED / "janelas.joblib",
)

joblib.dump(
    {
        "preprocessador": preprocessador,
        "colunas_numericas": colunas_numericas,
        "colunas_categoricas": colunas_categoricas,
    },
    PROCESSED / "preprocessamento.joblib",
)

print("Artefatos salvos em:", PROCESSED)
for arquivo in sorted(PROCESSED.glob("*.joblib")):
    print(f"  {arquivo.name}: {arquivo.stat().st_size / 1e6:.1f} MB")